# LANGFENS — RQ1-TRN-02B: Official Arm C1 Training on Kaggle Tesla T4x2
Official execution of Treatment Arm C1 (p=0.25) on Kaggle Tesla T4x2 GPU (`KAGGLE_T4_CUDA_FP16_AMP`).

## Scientific Protocol Invariants
- Architecture: Two-head Wav2Vec2 + Binary MDD (frozen backbone `facebook/wav2vec2-base-960h`)
- Arm: Treatment C1 (p=0.25, whole-phone binary prompt dropout)
- Budget: strictly matched 6 epochs / 3,600 successful updates max
- Precision: CUDA FP16 AMP (initial scale 4096.0)
- Seed: 1337 (cold reset AdamW, cosine schedule with 100 warmup updates)
- Data: Authorized Fold 1 Train+Val only (18 speakers; sealed test strictly excluded)
- Remote Durability: Per-epoch atomic mirror to private Hugging Face repository with SHA-256 readback

In [ ]:
# Phase 1: Hardware & Device Verification
!nvidia-smi
import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")

In [ ]:
# Phase 2: Clone and Checkout Pinned Immutable Training Commit
import subprocess, os, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    !git clone https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git /kaggle/working/repo

%cd /kaggle/working/repo
!git fetch --all
target_branch = "review/rq1-trn-02b-kaggle-official-runs"
target_commit = os.environ.get("TARGET_COMMIT", None)
if target_commit:
    !git checkout --detach {target_commit}
else:
    !git checkout --detach origin/{target_branch}
head = !git rev-parse HEAD
print(f"[GIT] Verified Training Commit HEAD: {head[0]}")
status = !git status --porcelain
assert len(status) == 0 or status == [""], f"Dirty tree: {status}"

sys.path.insert(0, "/kaggle/working/repo/src")
sys.path.insert(0, "/kaggle/working/repo")

In [ ]:
# Phase 3: Stage Authorized Dataset & Baseline R Weights
from pathlib import Path
import os

token_candidates = [
    Path("/kaggle/input/langfens-rq1-fold1-staging/hf_token.txt"),
    *list(Path("/kaggle/input").rglob("hf_token.txt")),
    Path("/kaggle/working/hf_token.txt"),
    Path("/kaggle/working/.hf_token"),
    Path("/kaggle/working/deploy/kaggle/hf_token.txt"),
    Path("/kaggle/working/repo/deploy/kaggle/hf_token.txt"),
]
for tp in token_candidates:
    if tp.is_file():
        os.environ["HF_TOKEN"] = tp.read_text().strip()
        print(f"[AUTH] Loaded HF_TOKEN from {tp}")
        break

print("[INPUT AUDIT]")
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 3

# Preflight staging and verified input checks
!python3 audit_scripts/kaggle_stage_and_verify.py --config configs/rq1_trn_02b_c1_kaggle.yaml --arm C1 --working-dir /kaggle/working --remote-mirror-backend huggingface --remote-mirror-target minkhoaa/langfens-rq1-trn-02b-checkpoints

In [ ]:
# Phase 4: Execute Official Matched-Budget Training: Arm C1
import os
head = !git rev-parse HEAD
commit_sha = head[0].strip()
token = os.environ.get("HF_TOKEN", "")
token_arg = f"--remote-mirror-token {token}" if token else ""

!python3 audit_scripts/kaggle_train_exec.py --config configs/rq1_trn_02b_c1_kaggle.yaml --arm C1 --working-dir /kaggle/working --remote-mirror-backend huggingface --remote-mirror-target minkhoaa/langfens-rq1-trn-02b-checkpoints --commit-sha {commit_sha} {token_arg}

In [ ]:
# Phase 5: Post-Training Final Endpoint Verification
from audit_scripts.kaggle_train_exec import classify_kaggle_training_endpoint
res = classify_kaggle_training_endpoint("/kaggle/working/checkpoints/RQ1-TRN-02B/C1", expected_arm="C1")
print("[FINAL ENDPOINT STATUS]:", res["status"])
print("[DETAILS]:", res.get("message"))
assert res["status"] == "COMPLETED_3600_UPDATES", f"Endpoint not completed: {res}"